# 画像生成のしくみをのぞく / Open-source image generation (Day 1 実習)

> **English version / 英語版はこちら**: [Open in Colab](https://colab.research.google.com/github/itoksk/summer-ai-materials/blob/main/materials/notebooks/image_gen_advanced_en.ipynb)

Day 1 の画像コンテストで使った Gemini は「完成品のサービス」。ここでは **オープンなモデル(Stable Diffusion)を自分の手で動かして**、中のつまみを直接さわります。

このノートは公開リポジトリ **github.com/itoksk/summer-ai-materials** の `materials/notebooks/` にあり、「Open in Colab」で GitHub から複製されました。

**準備 / Setup**
1. メニュー「ファイル → ドライブにコピーを保存」(編集と画像が自分のドライブに残ります)
2. メニュー「ランタイム → ランタイムのタイプを変更」で **T4 GPU** を選ぶ(無料枠でOK)
3. セルを上から順に実行。**つまみは各セル上の「フォーム」で変えられます**(コードは「Show code」で見られます)

**約束 / Rules**
- 実在の人物・友だち・有名人の名前や顔は生成しない(肖像権)
- 実在のキャラクター(アニメ・ゲーム)をそのまま出さない — 著作権ワークで考えたとおり
- AI生成を人作と偽る・なりすまし・ディープフェイクは作らない(犯罪になりえます)
- 作った画像には「created with Stable Diffusion」と添え、公開・商用の前にライセンスを確認する
- モデルのライセンスは CreativeML OpenRAIL-M / OpenRAIL++ 系(禁止用途が定められています)

(この教材は学校の授業「AI画像処理と著作権」のノートブックをもとにしています)

## Step 1 — 道具を入れる / Install

In [ ]:
# diffusers などを最新化(高画質モデル Animagine の長文プロンプト対応に新しめの diffusers が必要)
!pip -q install --upgrade diffusers transformers accelerate safetensors

## Step 2 — モデルを選んで読み込む / Pick a model and load it

何十億枚の画像で訓練済みのモデル(数GB)をダウンロードします。数分かかります。

**モデルは下のフォームの `MODEL` で選びます。** 2つの段があります:

| 段 | `MODEL` の値 | 特徴 |
|---|---|---|
| 基本(SD1.5) | `anime` / `base` / `photo` | **速い**・**安全フィルタON**・つまみが教科書どおり。まずはここから |
| 高画質(SDXL) | `anime-xl` / `photo-xl` | **きれいだが重い**・内蔵フィルタなし・書き方が少し違う |

**書き方(プロンプト)の違いに注意**
- `anime-xl`(Animagine XL 4.0) … 自然な文より **Danbooru風のタグ**で書くモデル。末尾の品質タグは自動で付きます。
- それ以外 … いつもの自然な英文でOK。
- → モデルを選ぶと、**そのモデルのおすすめプロンプト/設定が下に表示**されます(Step 3 のフォームにコピペ)。

**安全フィルタの話(ふりかえりの実例)**
- 基本段(SD1.5)は **NSFWフィルタを入れた状態**で読み込みます。引っかかると画像が**真っ黒**で返ります = バグではなくフィルタが働いた合図。
- 高画質段(SDXL)には、この内蔵フィルタが**付きません**。
- 「同じオープンモデルでも、サービス(Gemini)はなぜフィルタを足すのか?」— 最後のふりかえりで確認します。

**重さ / VRAM** — SDXLは1枚が重い。無料T4で `Out of memory` が出たら → 基本段に戻すか、下の `steps` を下げる。

In [ ]:
#@title ① モデルを選んで読み込む / Pick a model & load  { display-mode: "form" }
#@markdown 基本(速い・フィルタON): **anime / base / photo**　｜　高画質(SDXL・重い・フィルタなし): **anime-xl / photo-xl**
MODEL = "anime"  #@param ["anime", "base", "photo", "anime-xl", "photo-xl"]
#@markdown 変えたら左の ▶ を押して読み込み直す。

import torch
from diffusers import (
    StableDiffusionPipeline, StableDiffusionXLPipeline,
    DPMSolverMultistepScheduler, AutoencoderKL,
)

# GPUが無いと to("cuda") で止まるので最初に確認(なければ ランタイム→ランタイムのタイプを変更→T4 GPU)
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPUが見つかりません。Colab上部メニュー[ランタイム]→[ランタイムのタイプを変更]"
        "→ハードウェアアクセラレータで【T4 GPU】を選び、保存してから、もう一度実行してください。")

# --- モデル表(リポジトリ名と、各モデルのおすすめ設定) ---
ANIME_XL_NEG = ("lowres, bad anatomy, bad hands, text, error, missing finger, "
                "extra digits, fewer digits, cropped, worst quality, low quality, "
                "low score, bad score, average score, signature, watermark, username, blurry")
PHOTO_XL_NEG = ("worst quality, low quality, illustration, 3d, 2d, painting, cartoon, "
                "sketch, deformed iris, deformed pupils, bad anatomy, extra fingers, "
                "fused fingers, blurry, text, watermark")
SD15_PHOTO_NEG = ("cartoon, anime, sketch, drawing, 3d, render, worst quality, low quality, "
                  "bad anatomy, extra fingers, mutated hands, blurry, text, watermark")

MODELS = {
    # ---- 基本段: SD1.5(速い・フィルタON・自然文OK) ----
    "anime": {"arch": "sd15", "repo": "stablediffusionapi/anything-v5",
              "prompt": "a cozy ramen shop on a rainy night, neon signs, watercolor style",
              "neg": "low quality, blurry, text, watermark", "suffix": "",
              "steps": 25, "cfg": 7.0, "w": 512, "h": 512},
    "base":  {"arch": "sd15", "repo": "stable-diffusion-v1-5/stable-diffusion-v1-5",
              "prompt": "a cozy ramen shop on a rainy night, neon signs, watercolor style",
              "neg": "low quality, blurry, text, watermark", "suffix": "",
              "steps": 25, "cfg": 7.0, "w": 512, "h": 512},
    "photo": {"arch": "sd15", "repo": "SG161222/Realistic_Vision_V5.1_noVAE", "vae": "mse",
              "prompt": "a cozy ramen shop on a rainy night, neon signs, photo, 35mm",
              "neg": SD15_PHOTO_NEG, "suffix": "",
              "steps": 25, "cfg": 7.0, "w": 512, "h": 512},
    # ---- 高画質段: SDXL(きれい・重い・内蔵フィルタなし) ----
    "anime-xl": {"arch": "sdxl", "repo": "cagliostrolab/animagine-xl-4.0", "lpw": True,
                 "prompt": "no humans, ramen shop interior, neon signs, rainy night, "
                           "window reflection, steam, warm lighting, detailed background",
                 "neg": ANIME_XL_NEG,
                 "suffix": ", masterpiece, high score, great score, absurdres",
                 "steps": 28, "cfg": 5.5, "w": 1024, "h": 1024},
    "photo-xl": {"arch": "sdxl", "repo": "SG161222/RealVisXL_V5.0",
                 "prompt": "a cozy ramen shop on a rainy night, neon signs reflecting on wet "
                           "pavement, cinematic lighting, photorealistic, 35mm",
                 "neg": PHOTO_XL_NEG, "suffix": "",
                 "steps": 30, "cfg": 5.0, "w": 1024, "h": 1024},
}

cfg = MODELS[MODEL]
print("loading:", MODEL, "->", cfg["repo"])

if cfg["arch"] == "sd15":
    # SD1.5: NSFW安全フィルタを必ず付けて読み込む(学校で使うため)
    from transformers import CLIPImageProcessor
    from diffusers.pipelines.stable_diffusion.safety_checker import StableDiffusionSafetyChecker
    checker = StableDiffusionSafetyChecker.from_pretrained(
        "CompVis/stable-diffusion-safety-checker", torch_dtype=torch.float16)
    extractor = CLIPImageProcessor.from_pretrained("CompVis/stable-diffusion-safety-checker")
    kw = dict(torch_dtype=torch.float16, safety_checker=checker, feature_extractor=extractor)
    if cfg.get("vae") == "mse":   # *_noVAE モデルは VAE を足さないと色がくすむ
        kw["vae"] = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse",
                                                  torch_dtype=torch.float16)
    pipe = StableDiffusionPipeline.from_pretrained(cfg["repo"], **kw)
else:
    # SDXL: 高画質。内蔵フィルタは付かない(Step 2 の説明を参照)
    kw = dict(torch_dtype=torch.float16, use_safetensors=True, add_watermarker=False)
    if cfg.get("lpw"):            # 長文・タグ・強調プロンプトに強いパイプライン
        kw["custom_pipeline"] = "lpw_stable_diffusion_xl"
    pipe = StableDiffusionXLPipeline.from_pretrained(cfg["repo"], **kw)

# スケジューラ(ノイズの消し方)。Karras 版にすると質感が安定します
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True, algorithm_type="dpmsolver++")
pipe = pipe.to("cuda")

HAS_FILTER = (cfg["arch"] == "sd15")
print("ready:", cfg["repo"], "| 安全フィルタ:", "ON" if HAS_FILTER else "なし(SDXL)")
print()
print("--- このモデルのおすすめ(Step 3 のフォームにコピペ) ---")
print("prompt   :", cfg["prompt"])
print("negative :", cfg["neg"])
print(f"steps={cfg['steps']}  guidance={cfg['cfg']}  size={cfg['w']}x{cfg['h']}")
if cfg["suffix"]:
    print("品質タグ", repr(cfg["suffix"]), "は自動で末尾に付きます")

## Step 3 — 生成する / Generate

つまみの意味:
- `prompt`: 何を描くか(**英語**で。Gemini に翻訳してもらってOK)
- `negative_prompt`: 入れたくないもの
- `seed`: サイコロの目。**同じ seed + 同じ言葉 = 同じ絵**
- `guidance`: 言葉にどれだけ従うか(基本7前後 / SDXLは5前後が定番)
- `steps`: ノイズから絵にする回数(25〜28前後)

下のフォームで変えて ▶ を押すだけ。**選んだモデルのおすすめ**は Step 2 の出力に出ています(コピペ可)。

In [ ]:
#@title ② 生成する / Generate  { display-mode: "form" }
#@markdown 英語で(Geminiに訳してOK)。同じ seed + 同じ言葉 = 同じ絵。SDXL(anime-xl/photo-xl)は guidance 5 前後がきれい。
prompt = "a cozy ramen shop on a rainy night, neon signs, watercolor style"  #@param {type:"string"}
negative_prompt = "low quality, blurry, text, watermark"  #@param {type:"string"}
seed = 42  #@param {type:"integer"}
steps = 25  #@param {type:"slider", min:10, max:40, step:1}
guidance = 7.0  #@param {type:"slider", min:1, max:15, step:0.5}

full_prompt = prompt + cfg["suffix"]          # 品質タグ(モデルごと)を自動付与
generator = torch.Generator("cuda").manual_seed(seed)
image = pipe(full_prompt, negative_prompt=negative_prompt,
             num_inference_steps=steps, guidance_scale=guidance,
             width=cfg["w"], height=cfg["h"], generator=generator).images[0]
# 画像が真っ黒なら → 安全フィルタが働いた合図(SD1.5のみ)。言葉を変えて再挑戦
image

## Step 4 — 実験 / Experiments

1. **seed を固定して言葉だけ変える** — `watercolor style` を `pixel art` にすると?
2. **言葉を固定して seed だけ変える** — 同じ注文でも違う絵になる
3. `guidance` を 2 と 15 にしてみる — 従いすぎると絵はどうなる?

(Day 1 コンテストの「言葉で絵を支配する」感覚を、今度はつまみのレベルで)

※ この実験が成り立つのは、つまみが素直に効く**基本段**と**標準SDXL**だから。4〜6ステップで一気に描く「高速版(Lightning/Turbo)」だと guidance が効かず、この実験は壊れます。

In [ ]:
#@title ③ 実験: つまみを変えてもう1枚 / Experiment  { display-mode: "form" }
#@markdown seed を固定して言葉だけ変える / 言葉を固定して seed を変える / guidance を 2 と 15 で比べる
prompt = "a cozy ramen shop on a rainy night, neon signs, pixel art"  #@param {type:"string"}
seed = 42  #@param {type:"integer"}
steps = 25  #@param {type:"slider", min:10, max:40, step:1}
guidance = 7.0  #@param {type:"slider", min:1, max:15, step:0.5}

full_prompt = prompt + cfg["suffix"]
generator = torch.Generator("cuda").manual_seed(seed)
pipe(full_prompt, negative_prompt=cfg["neg"],
     num_inference_steps=steps, guidance_scale=guidance,
     width=cfg["w"], height=cfg["h"], generator=generator).images[0]

## ふりかえり / Reflection

- このモデルは「誰の絵」で訓練された? 描いた人は同意した? — 著作権ワークの問いを、今日は**自分が生成する側**として考え直す
- Gemini のようなサービスと、自分で動かすオープンモデルの違いは? (安全フィルター、責任、自由度)
  - 今日は **基本段=フィルタON / 高画質段(SDXL)=フィルタなし** を実際に切り替えた。サービスがフィルタを足すのはなぜ?
- 生成した絵は「あなたの作品」? AI の作品? 訓練データの作者の作品?